## **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

if IS_COLAB:
    !pip install optuna

Repo already exists — pulling latest changes
Already up to date.


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [3]:
import numpy as np
import optuna

from Challenge.paths import load_cv_folds, generate_submission, MODEL_DIR
from Challenge.hyper_tuning import ModelOptimizer
from Challenge.utils import evaluate_recommender

Running on local — storage at: /home/luigi/RecSys


/home/luigi/RecSys/Challenge/hyper_tuning.py:75: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_study(self, study_name, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):
/home/luigi/RecSys/Challenge/hyper_tuning.py:107: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_and_optimize_study(self, study_name, objective_function, n_trials=50, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):


## **Load Data**

In [4]:
# Load datasets
folds = load_cv_folds(k=5)

## **Hyperparameter search**

In [5]:
from Recommenders.Neural.MultVAE_PyTorch_Recommender import MultVAERecommender_PyTorch_OptimizerMask

optimizer = ModelOptimizer("MultVAE")

STUDY_NAME = MultVAERecommender_PyTorch_OptimizerMask.RECOMMENDER_NAME

In [6]:
URM_train, URM_validation = folds[0]  # Too slow to test on all folds

def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "epochs": optuna_trial.suggest_int("epochs", 10, 50, step=10),
        "batch_size": optuna_trial.suggest_categorical("batch_size", [256, 512, 1024]),
        "total_anneal_steps": optuna_trial.suggest_int("total_anneal_steps", 100000, 500000, step=100000),
        "learning_rate": optuna_trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "l2_reg": optuna_trial.suggest_float("l2_reg", 1e-5, 1e-1, log=True),
        "dropout": optuna_trial.suggest_float("dropout", 0.0, 0.7, step=0.1),
        "anneal_cap": optuna_trial.suggest_float("anneal_cap", 0.0, 0.5, step=0.1),
        "sgd_mode": optuna_trial.suggest_categorical("sgd_mode", ["adam", "sgd", "adagrad"]),
        "encoding_size": optuna_trial.suggest_int("encoding_size", 20, 200, step=20),
        "next_layer_size_multiplier": optuna_trial.suggest_int("next_layer_size_multiplier", 2, 4),
        "max_parameters": optuna_trial.suggest_int("max_parameters", 1e5, 1e7, step=1e5),
        "max_n_hidden_layers": optuna_trial.suggest_int("max_n_hidden_layers", 1, 5)
    }

    # Train the recommender
    recommender_instance = MultVAERecommender_PyTorch_OptimizerMask(URM_train)
    recommender_instance.fit(**params)
        
    # Evaluate
    score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)

    # Log folds performance
    optimizer.log_folds([score], params)

    # Return the mean CV score for the fully completed trial
    return score

In [7]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=10
)

[I 2025-11-28 20:25:36,834] A new study created in RDB with name: MultVAERecommender_PyTorch


  0%|          | 0/10 [00:00<?, ?it/s]

/home/luigi/RecSys/Recommenders/Neural/MultVAE_PyTorch_Recommender.py:330: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  user_batch_tensor = torch.sparse_csr_tensor(user_batch_tensor.indptr,


MultVAERecommender_PyTorch: Epoch 1 of 10. Elapsed time 0.82 sec
MultVAERecommender_PyTorch: Epoch 2 of 10. Elapsed time 1.21 sec
MultVAERecommender_PyTorch: Epoch 3 of 10. Elapsed time 1.60 sec
MultVAERecommender_PyTorch: Epoch 4 of 10. Elapsed time 2.00 sec
MultVAERecommender_PyTorch: Epoch 5 of 10. Elapsed time 2.38 sec
MultVAERecommender_PyTorch: Epoch 6 of 10. Elapsed time 2.78 sec
MultVAERecommender_PyTorch: Epoch 7 of 10. Elapsed time 3.17 sec
MultVAERecommender_PyTorch: Epoch 8 of 10. Elapsed time 3.56 sec
MultVAERecommender_PyTorch: Epoch 9 of 10. Elapsed time 3.95 sec
MultVAERecommender_PyTorch: Epoch 10 of 10. Elapsed time 4.33 sec
MultVAERecommender_PyTorch: Terminating at epoch 10. Elapsed time 4.34 sec


Eval Batches: 100%|██████████| 28/28 [00:02<00:00, 13.26it/s]


...

[I 2025-11-28 20:28:50,322] Trial 7 finished with value: 0.10459261849305398 and parameters: {'epochs': 50, 'batch_size': 1024, 'total_anneal_steps': 500000, 'learning_rate': 0.00010459464980606996, 'l2_reg': 0.027082008474762814, 'dropout': 0.2, 'anneal_cap': 0.1, 'sgd_mode': 'sgd', 'encoding_size': 80, 'next_layer_size_multiplier': 3, 'max_parameters': 4200000, 'max_n_hidden_layers': 3}. Best is trial 5 with value: 0.2284142696648038.
MultVAERecommender_PyTorch: Epoch 1 of 50. Elapsed time 0.61 sec
MultVAERecommender_PyTorch: Epoch 2 of 50. Elapsed time 1.21 sec
MultVAERecommender_PyTorch: Epoch 3 of 50. Elapsed time 1.81 sec
MultVAERecommender_PyTorch: Epoch 4 of 50. Elapsed time 2.34 sec
MultVAERecommender_PyTorch: Epoch 5 of 50. Elapsed time 2.91 sec
MultVAERecommender_PyTorch: Epoch 6 of 50. Elapsed time 3.45 sec
MultVAERecommender_PyTorch: Epoch 7 of 50. Elapsed time 3.96 sec
MultVAERecommender_PyTorch: Epoch 8 of 50. Elapsed time 4.47 sec
MultVAERecommender_PyTorch: Epoch 9 of 

Eval Batches: 100%|██████████| 28/28 [00:03<00:00,  8.70it/s]


[I 2025-11-28 20:29:23,039] Trial 8 finished with value: 0.22258824763392943 and parameters: {'epochs': 50, 'batch_size': 256, 'total_anneal_steps': 200000, 'learning_rate': 0.004141587961200636, 'l2_reg': 0.004395808831464697, 'dropout': 0.1, 'anneal_cap': 0.4, 'sgd_mode': 'adam', 'encoding_size': 60, 'next_layer_size_multiplier': 4, 'max_parameters': 6600000, 'max_n_hidden_layers': 1}. Best is trial 5 with value: 0.2284142696648038.
MultVAERecommender_PyTorch: Epoch 1 of 10. Elapsed time 0.41 sec
MultVAERecommender_PyTorch: Epoch 2 of 10. Elapsed time 0.75 sec
MultVAERecommender_PyTorch: Epoch 3 of 10. Elapsed time 1.07 sec
MultVAERecommender_PyTorch: Epoch 4 of 10. Elapsed time 1.40 sec
MultVAERecommender_PyTorch: Epoch 5 of 10. Elapsed time 1.77 sec
MultVAERecommender_PyTorch: Epoch 6 of 10. Elapsed time 2.14 sec
MultVAERecommender_PyTorch: Epoch 7 of 10. Elapsed time 2.50 sec
MultVAERecommender_PyTorch: Epoch 8 of 10. Elapsed time 2.87 sec
MultVAERecommender_PyTorch: Epoch 9 of 10

Eval Batches: 100%|██████████| 28/28 [00:01<00:00, 15.38it/s]


[I 2025-11-28 20:29:28,556] Trial 9 finished with value: 0.10834089439669933 and parameters: {'epochs': 10, 'batch_size': 512, 'total_anneal_steps': 500000, 'learning_rate': 0.0030625501296691494, 'l2_reg': 0.0004441737214360341, 'dropout': 0.1, 'anneal_cap': 0.0, 'sgd_mode': 'adagrad', 'encoding_size': 20, 'next_layer_size_multiplier': 3, 'max_parameters': 2900000, 'max_n_hidden_layers': 1}. Best is trial 5 with value: 0.2284142696648038.

Study statistics: 
  Number of finished trials:  10
  Number of pruned trials:  0
  Number of complete trials:  10

Best Value: 0.2284142696648038
Best Params: {'epochs': 30, 'batch_size': 256, 'total_anneal_steps': 300000, 'learning_rate': 0.001703807063909963, 'l2_reg': 0.018618931526378817, 'dropout': 0.5, 'anneal_cap': 0.1, 'sgd_mode': 'adam', 'encoding_size': 160, 'next_layer_size_multiplier': 3, 'max_parameters': 9800000, 'max_n_hidden_layers': 2}


In [8]:
optuna.visualization.plot_optimization_history(optuna_study)

In [9]:
optuna.visualization.plot_param_importances(optuna_study)

In [10]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Hyperparameter tuning**

In [11]:
STUDY_NAME = MultVAERecommender_PyTorch_OptimizerMask.RECOMMENDER_NAME + "_v1"

In [12]:
URM_train, URM_validation = folds[0]  # Too slow to test on all folds

def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "epochs": optuna_trial.suggest_int("epochs", 50, 100, step=10),
        "batch_size": optuna_trial.suggest_categorical("batch_size", [256, 512, 1024]),
        "total_anneal_steps": optuna_trial.suggest_int("total_anneal_steps", 100000, 500000, step=100000),
        "learning_rate": optuna_trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "l2_reg": optuna_trial.suggest_float("l2_reg", 1e-5, 1e-1, log=True),
        "dropout": optuna_trial.suggest_float("dropout", 0.0, 0.7, step=0.1),
        "anneal_cap": optuna_trial.suggest_float("anneal_cap", 0.0, 0.5, step=0.1),
        "sgd_mode": "adam",
        "encoding_size": optuna_trial.suggest_int("encoding_size", 20, 200, step=20),
        "next_layer_size_multiplier": optuna_trial.suggest_int("next_layer_size_multiplier", 2, 4),
        "max_parameters": optuna_trial.suggest_int("max_parameters", 1e5, 1e7, step=1e5),
        "max_n_hidden_layers": optuna_trial.suggest_int("max_n_hidden_layers", 1, 5)
    }

    # Train the recommender
    recommender_instance = MultVAERecommender_PyTorch_OptimizerMask(URM_train)
    recommender_instance.fit(**params)
        
    # Evaluate
    score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)

    # Log folds performance
    optimizer.log_folds([score], params)

    # Return the mean CV score for the fully completed trial
    return score

In [13]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=20
)

[I 2025-11-28 20:33:07,427] A new study created in RDB with name: MultVAERecommender_PyTorch_v1


  0%|          | 0/20 [00:00<?, ?it/s]

MultVAERecommender_PyTorch: Epoch 1 of 70. Elapsed time 0.99 sec
MultVAERecommender_PyTorch: Epoch 2 of 70. Elapsed time 2.11 sec
MultVAERecommender_PyTorch: Epoch 3 of 70. Elapsed time 2.97 sec
MultVAERecommender_PyTorch: Epoch 4 of 70. Elapsed time 4.03 sec
MultVAERecommender_PyTorch: Epoch 5 of 70. Elapsed time 4.86 sec
MultVAERecommender_PyTorch: Epoch 6 of 70. Elapsed time 5.70 sec
MultVAERecommender_PyTorch: Epoch 7 of 70. Elapsed time 6.57 sec
MultVAERecommender_PyTorch: Epoch 8 of 70. Elapsed time 7.51 sec
MultVAERecommender_PyTorch: Epoch 9 of 70. Elapsed time 8.39 sec
MultVAERecommender_PyTorch: Epoch 10 of 70. Elapsed time 9.34 sec
MultVAERecommender_PyTorch: Epoch 11 of 70. Elapsed time 10.21 sec
MultVAERecommender_PyTorch: Epoch 12 of 70. Elapsed time 11.18 sec
MultVAERecommender_PyTorch: Epoch 13 of 70. Elapsed time 12.19 sec
MultVAERecommender_PyTorch: Epoch 14 of 70. Elapsed time 13.23 sec
MultVAERecommender_PyTorch: Epoch 15 of 70. Elapsed time 14.22 sec
MultVAERecomme

Eval Batches: 100%|██████████| 28/28 [00:01<00:00, 14.09it/s]


[I 2025-11-28 20:34:10,365] Trial 0 finished with value: 0.2433429442053331 and parameters: {'epochs': 70, 'batch_size': 256, 'total_anneal_steps': 400000, 'learning_rate': 0.00029459938907104116, 'l2_reg': 0.011220221556617532, 'dropout': 0.4, 'anneal_cap': 0.0, 'encoding_size': 180, 'next_layer_size_multiplier': 3, 'max_parameters': 5800000, 'max_n_hidden_layers': 4}. Best is trial 0 with value: 0.2433429442053331.
MultVAERecommender_PyTorch: Epoch 1 of 50. Elapsed time 0.37 sec
MultVAERecommender_PyTorch: Epoch 2 of 50. Elapsed time 0.73 sec
MultVAERecommender_PyTorch: Epoch 3 of 50. Elapsed time 1.11 sec
MultVAERecommender_PyTorch: Epoch 4 of 50. Elapsed time 1.50 sec
MultVAERecommender_PyTorch: Epoch 5 of 50. Elapsed time 1.88 sec
MultVAERecommender_PyTorch: Epoch 6 of 50. Elapsed time 2.30 sec
MultVAERecommender_PyTorch: Epoch 7 of 50. Elapsed time 2.72 sec
MultVAERecommender_PyTorch: Epoch 8 of 50. Elapsed time 3.07 sec
MultVAERecommender_PyTorch: Epoch 9 of 50. Elapsed time 3.4

...

[I 2025-11-28 20:49:13,054] Trial 17 finished with value: 0.23718595268393497 and parameters: {'epochs': 80, 'batch_size': 1024, 'total_anneal_steps': 400000, 'learning_rate': 0.0007359288878046676, 'l2_reg': 0.00016926244057316318, 'dropout': 0.6000000000000001, 'anneal_cap': 0.30000000000000004, 'encoding_size': 180, 'next_layer_size_multiplier': 4, 'max_parameters': 2100000, 'max_n_hidden_layers': 3}. Best is trial 0 with value: 0.2433429442053331.
MultVAERecommender_PyTorch: Epoch 1 of 70. Elapsed time 0.76 sec
MultVAERecommender_PyTorch: Epoch 2 of 70. Elapsed time 1.49 sec
MultVAERecommender_PyTorch: Epoch 3 of 70. Elapsed time 2.24 sec
MultVAERecommender_PyTorch: Epoch 4 of 70. Elapsed time 3.01 sec
MultVAERecommender_PyTorch: Epoch 5 of 70. Elapsed time 3.79 sec
MultVAERecommender_PyTorch: Epoch 6 of 70. Elapsed time 4.56 sec
MultVAERecommender_PyTorch: Epoch 7 of 70. Elapsed time 5.27 sec
MultVAERecommender_PyTorch: Epoch 8 of 70. Elapsed time 5.98 sec
MultVAERecommender_PyTor

Eval Batches: 100%|██████████| 28/28 [00:02<00:00, 10.22it/s]


[I 2025-11-28 20:50:09,874] Trial 18 finished with value: 0.20884215993380886 and parameters: {'epochs': 70, 'batch_size': 256, 'total_anneal_steps': 500000, 'learning_rate': 0.00010682622130755165, 'l2_reg': 0.030290274178871923, 'dropout': 0.1, 'anneal_cap': 0.2, 'encoding_size': 140, 'next_layer_size_multiplier': 2, 'max_parameters': 3500000, 'max_n_hidden_layers': 5}. Best is trial 0 with value: 0.2433429442053331.
MultVAERecommender_PyTorch: Epoch 1 of 60. Elapsed time 0.67 sec
MultVAERecommender_PyTorch: Epoch 2 of 60. Elapsed time 1.34 sec
MultVAERecommender_PyTorch: Epoch 3 of 60. Elapsed time 1.98 sec
MultVAERecommender_PyTorch: Epoch 4 of 60. Elapsed time 2.63 sec
MultVAERecommender_PyTorch: Epoch 5 of 60. Elapsed time 3.27 sec
MultVAERecommender_PyTorch: Epoch 6 of 60. Elapsed time 3.92 sec
MultVAERecommender_PyTorch: Epoch 7 of 60. Elapsed time 4.58 sec
MultVAERecommender_PyTorch: Epoch 8 of 60. Elapsed time 5.21 sec
MultVAERecommender_PyTorch: Epoch 9 of 60. Elapsed time 5

Eval Batches: 100%|██████████| 28/28 [00:02<00:00, 11.24it/s]


[I 2025-11-28 20:50:51,071] Trial 19 finished with value: 0.22733190786091428 and parameters: {'epochs': 60, 'batch_size': 512, 'total_anneal_steps': 400000, 'learning_rate': 0.0016842505370521798, 'l2_reg': 0.001236793382400099, 'dropout': 0.4, 'anneal_cap': 0.0, 'encoding_size': 180, 'next_layer_size_multiplier': 3, 'max_parameters': 6000000, 'max_n_hidden_layers': 4}. Best is trial 0 with value: 0.2433429442053331.

Study statistics: 
  Number of finished trials:  20
  Number of pruned trials:  0
  Number of complete trials:  20

Best Value: 0.2433429442053331
Best Params: {'epochs': 70, 'batch_size': 256, 'total_anneal_steps': 400000, 'learning_rate': 0.00029459938907104116, 'l2_reg': 0.011220221556617532, 'dropout': 0.4, 'anneal_cap': 0.0, 'encoding_size': 180, 'next_layer_size_multiplier': 3, 'max_parameters': 5800000, 'max_n_hidden_layers': 4}


In [14]:
optuna.visualization.plot_optimization_history(optuna_study)

In [15]:
optuna.visualization.plot_param_importances(optuna_study)

In [16]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **3**

In [ ]:
STUDY_NAME = MultVAERecommender_PyTorch_OptimizerMask.RECOMMENDER_NAME + "_v2"

In [17]:
URM_train, URM_validation = folds[0]  # Too slow to test on all folds

"""
Best Value: 0.2433429442053331
Best Params: {'epochs': 70, 'batch_size': 256, 'total_anneal_steps': 400000, 'learning_rate': 0.00029459938907104116, 'l2_reg': 0.011220221556617532, 'dropout': 0.4, 'anneal_cap': 0.0, 'encoding_size': 180, 'next_layer_size_multiplier': 3, 'max_parameters': 5800000, 'max_n_hidden_layers': 4}
"""

def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "epochs": 70,
        "batch_size": 256,
        "total_anneal_steps": optuna_trial.suggest_int("total_anneal_steps", 300000, 600000, step=50000),
        "learning_rate": optuna_trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True),
        "l2_reg": optuna_trial.suggest_float("l2_reg", 1e-3, 2e-1, log=True),
        "dropout": optuna_trial.suggest_float("dropout", 0.3, 0.6, step=0.1),
        "anneal_cap": optuna_trial.suggest_float("anneal_cap", 0.0, 0.2, step=0.1),
        "sgd_mode": "adam",
        "encoding_size": optuna_trial.suggest_int("encoding_size", 150, 250, step=10),
        "next_layer_size_multiplier": optuna_trial.suggest_int("next_layer_size_multiplier", 2, 4),
        "max_parameters": optuna_trial.suggest_int("max_parameters", 1e6, 1e7, step=1e5),
        "max_n_hidden_layers": optuna_trial.suggest_int("max_n_hidden_layers", 3, 5)
    }

    # Train the recommender
    recommender_instance = MultVAERecommender_PyTorch_OptimizerMask(URM_train)
    recommender_instance.fit(**params)
        
    # Evaluate
    score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)

    # Log folds performance
    optimizer.log_folds([score], params)

    # Return the mean CV score for the fully completed trial
    return score

In [18]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=50
)

[I 2025-11-28 20:57:52,665] Using an existing study with name 'MultVAERecommender_PyTorch_v1' instead of creating a new one.


  0%|          | 0/50 [00:00<?, ?it/s]

MultVAERecommender_PyTorch: Epoch 1 of 70. Elapsed time 1.05 sec
MultVAERecommender_PyTorch: Epoch 2 of 70. Elapsed time 2.06 sec
MultVAERecommender_PyTorch: Epoch 3 of 70. Elapsed time 3.03 sec
MultVAERecommender_PyTorch: Epoch 4 of 70. Elapsed time 4.37 sec
MultVAERecommender_PyTorch: Epoch 5 of 70. Elapsed time 5.41 sec
MultVAERecommender_PyTorch: Epoch 6 of 70. Elapsed time 6.74 sec
MultVAERecommender_PyTorch: Epoch 7 of 70. Elapsed time 7.98 sec
MultVAERecommender_PyTorch: Epoch 8 of 70. Elapsed time 9.20 sec
MultVAERecommender_PyTorch: Epoch 9 of 70. Elapsed time 10.41 sec
MultVAERecommender_PyTorch: Epoch 10 of 70. Elapsed time 11.54 sec
MultVAERecommender_PyTorch: Epoch 11 of 70. Elapsed time 12.76 sec
MultVAERecommender_PyTorch: Epoch 12 of 70. Elapsed time 13.80 sec
MultVAERecommender_PyTorch: Epoch 13 of 70. Elapsed time 14.95 sec
MultVAERecommender_PyTorch: Epoch 14 of 70. Elapsed time 16.22 sec
MultVAERecommender_PyTorch: Epoch 15 of 70. Elapsed time 17.36 sec
MultVAERecom

Eval Batches: 100%|██████████| 28/28 [00:03<00:00,  7.34it/s]


[I 2025-11-28 20:59:12,050] Trial 20 finished with value: 0.15893544243403243 and parameters: {'total_anneal_steps': 600000, 'learning_rate': 3.41442374988798e-05, 'l2_reg': 0.007990682475135505, 'dropout': 0.5, 'anneal_cap': 0.0, 'encoding_size': 250, 'next_layer_size_multiplier': 3, 'max_parameters': 5100000, 'max_n_hidden_layers': 4}. Best is trial 0 with value: 0.2433429442053331.
MultVAERecommender_PyTorch: Epoch 1 of 70. Elapsed time 0.98 sec
MultVAERecommender_PyTorch: Epoch 2 of 70. Elapsed time 1.93 sec
MultVAERecommender_PyTorch: Epoch 3 of 70. Elapsed time 2.84 sec
MultVAERecommender_PyTorch: Epoch 4 of 70. Elapsed time 3.72 sec
MultVAERecommender_PyTorch: Epoch 5 of 70. Elapsed time 4.64 sec
MultVAERecommender_PyTorch: Epoch 6 of 70. Elapsed time 5.53 sec
MultVAERecommender_PyTorch: Epoch 7 of 70. Elapsed time 6.43 sec
MultVAERecommender_PyTorch: Epoch 8 of 70. Elapsed time 7.40 sec
MultVAERecommender_PyTorch: Epoch 9 of 70. Elapsed time 8.34 sec
MultVAERecommender_PyTorch:

...

[I 2025-11-28 21:50:27,741] Trial 67 finished with value: 0.22180004443022508 and parameters: {'total_anneal_steps': 600000, 'learning_rate': 0.00014682918514519655, 'l2_reg': 0.03361791398982583, 'dropout': 0.4, 'anneal_cap': 0.0, 'encoding_size': 190, 'next_layer_size_multiplier': 3, 'max_parameters': 5700000, 'max_n_hidden_layers': 5}. Best is trial 26 with value: 0.24549235309817774.
MultVAERecommender_PyTorch: Epoch 1 of 70. Elapsed time 0.72 sec
MultVAERecommender_PyTorch: Epoch 2 of 70. Elapsed time 1.46 sec
MultVAERecommender_PyTorch: Epoch 3 of 70. Elapsed time 2.18 sec
MultVAERecommender_PyTorch: Epoch 4 of 70. Elapsed time 2.91 sec
MultVAERecommender_PyTorch: Epoch 5 of 70. Elapsed time 3.63 sec
MultVAERecommender_PyTorch: Epoch 6 of 70. Elapsed time 4.35 sec
MultVAERecommender_PyTorch: Epoch 7 of 70. Elapsed time 5.10 sec
MultVAERecommender_PyTorch: Epoch 8 of 70. Elapsed time 5.83 sec
MultVAERecommender_PyTorch: Epoch 9 of 70. Elapsed time 6.61 sec
MultVAERecommender_PyTor

Eval Batches: 100%|██████████| 28/28 [00:01<00:00, 16.42it/s]


[I 2025-11-28 21:51:20,553] Trial 68 finished with value: 0.2425274900105384 and parameters: {'total_anneal_steps': 550000, 'learning_rate': 0.00037457369941927196, 'l2_reg': 0.007699099957824215, 'dropout': 0.4, 'anneal_cap': 0.1, 'encoding_size': 170, 'next_layer_size_multiplier': 4, 'max_parameters': 6500000, 'max_n_hidden_layers': 5}. Best is trial 26 with value: 0.24549235309817774.
MultVAERecommender_PyTorch: Epoch 1 of 70. Elapsed time 0.88 sec
MultVAERecommender_PyTorch: Epoch 2 of 70. Elapsed time 1.75 sec
MultVAERecommender_PyTorch: Epoch 3 of 70. Elapsed time 2.62 sec
MultVAERecommender_PyTorch: Epoch 4 of 70. Elapsed time 3.51 sec
MultVAERecommender_PyTorch: Epoch 5 of 70. Elapsed time 4.38 sec
MultVAERecommender_PyTorch: Epoch 6 of 70. Elapsed time 5.24 sec
MultVAERecommender_PyTorch: Epoch 7 of 70. Elapsed time 6.11 sec
MultVAERecommender_PyTorch: Epoch 8 of 70. Elapsed time 6.99 sec
MultVAERecommender_PyTorch: Epoch 9 of 70. Elapsed time 7.86 sec
MultVAERecommender_PyTor

Eval Batches: 100%|██████████| 28/28 [00:01<00:00, 18.16it/s]


[I 2025-11-28 21:52:23,243] Trial 69 finished with value: 0.24008501001310523 and parameters: {'total_anneal_steps': 400000, 'learning_rate': 0.00020587351580808022, 'l2_reg': 0.17852125409650302, 'dropout': 0.4, 'anneal_cap': 0.0, 'encoding_size': 230, 'next_layer_size_multiplier': 3, 'max_parameters': 6100000, 'max_n_hidden_layers': 5}. Best is trial 26 with value: 0.24549235309817774.

Study statistics: 
  Number of finished trials:  70
  Number of pruned trials:  0
  Number of complete trials:  70

Best Value: 0.24549235309817774
Best Params: {'total_anneal_steps': 600000, 'learning_rate': 0.00030599089006292914, 'l2_reg': 0.0104723014135945, 'dropout': 0.4, 'anneal_cap': 0.1, 'encoding_size': 220, 'next_layer_size_multiplier': 3, 'max_parameters': 4500000, 'max_n_hidden_layers': 5}


In [19]:
optuna.visualization.plot_optimization_history(optuna_study)

In [22]:
optuna.visualization.plot_param_importances(optuna_study)

In [21]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Best Model**
- Best Value: 0.24549235309817774
Best Params: {'total_anneal_steps': 600000, 'learning_rate': 0.00030599089006292914, 'l2_reg': 0.0104723014135945, 'dropout': 0.4, 'anneal_cap': 0.1, 'encoding_size': 220, 'next_layer_size_multiplier': 3, 'max_parameters': 4500000, 'max_n_hidden_layers': 5}